# Comparação de eficiência de CNNs — ISIC 2019

Driver para rodar o estudo no Google Colab.

**Antes de começar:** `Runtime ▸ Change runtime type ▸ GPU`.

Rode as células na ordem. `results/`, `checkpoints/` e `figs/` são gravados
no seu Google Drive, então o progresso sobrevive a quedas de sessão.
O treino é **resumível**: se a sessão cair, re-execute a mesma célula de treino.

## 1. Setup — clona o repositório e instala dependências

In [ ]:
import os

REPO_URL = "https://github.com/pedruck/skin-cnn-efficiency"
REPO_DIR = "/content/skin-cnn-efficiency"

if not os.path.isdir(REPO_DIR):
    !git clone -q "$REPO_URL.git" "$REPO_DIR"
else:
    !cd "$REPO_DIR" && git pull -q

%cd $REPO_DIR
!pip install -q thop kagglehub pyyaml nvidia-ml-py
print('OK')

## 2. Google Drive para as saídas persistentes

`results/`, `checkpoints/` e `figs/` viram links para uma pasta no seu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/skin-cnn-efficiency'
for sub in ['results', 'checkpoints', 'figs']:
    os.makedirs(f'{DRIVE_ROOT}/{sub}', exist_ok=True)
    link = f'{REPO_DIR}/{sub}'
    if os.path.islink(link) or os.path.isdir(link):
        !rm -rf "$link"
    os.symlink(f'{DRIVE_ROOT}/{sub}', link)
print('saidas ->', DRIVE_ROOT)

## 3. Dataset ISIC 2019 (download via kagglehub)

Re-baixa a cada sessão (poucos GB). Se pedir credencial, faça upload do
`kaggle.json` (Kaggle ▸ Account ▸ Create New Token) em `/root/.config/kaggle/`.

In [ ]:
import kagglehub
DATA = kagglehub.dataset_download('salviohexia/isic-2019-skin-lesion-images-for-classification')
print('DATA =', DATA)

## 4. Treino

**Uma arquitetura por célula.** ~25–55 min cada numa T4 (ResNet-50 é a mais lenta).
Se a sessão cair, é só re-executar a célula: retoma da última época salva.

Ajuste as épocas em `config.yaml` (padrão: 20) ou com `--epochs`.

In [ ]:
!python -m src.train --model resnet50 --data "$DATA"

In [ ]:
!python -m src.train --model resnet18 --data "$DATA"

In [ ]:
!python -m src.train --model mobilenet_v2 --data "$DATA"

In [ ]:
!python -m src.train --model mobilenet_v3_large --data "$DATA"

## 5. Benchmark — avaliação no teste + métricas de eficiência

Gera `results/performance.csv`, `efficiency.csv`, `summary.csv`,
`environment.json` e `test_predictions_<modelo>.npz`.

In [ ]:
!python -m src.benchmark --data "$DATA"

## 6. Figuras

In [ ]:
!python -m src.plots --data "$DATA"

import glob
from IPython.display import Image, display
for f in sorted(glob.glob('figs/*.png')):
    print(f); display(Image(f))

## 7. Baixar artefatos

Também já estão no seu Drive em `MyDrive/skin-cnn-efficiency/`.

In [ ]:
!zip -qr /content/artefatos.zip results figs
from google.colab import files
files.download('/content/artefatos.zip')